# B2.3 · Tool design

**Function B — Product & Application Security → The Security Automation / Harness Engineer**  ·  *Security of AI*

Builds on **[B2.2 · Verify signals that don't lie](https://spbreed.github.io/cyber-commons/lessons/B2.2.html)**.

| | |
|---|---|
| Open-source tooling | kmcp |
| Open-weight models | — |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off.

## 1 · The concept


Tool design is security design, and it is stronger than anything you can put in
a prompt — because a tool's **signature decides what the model is able to ask
for**.

Two tools with identical underlying capability:

```python
read_file(path: str)              # the model can request any path
read_file(doc_id: Literal[...])   # the model can request one of five documents
```

Both may be safe if a path guard sits underneath. The difference appears when
the guard has a bug: the first tool *presents* the vulnerable surface, the
second never expresses it.

Three rules follow, and they compose:

1. **Enumerate rather than accept free text** wherever the real requirement is a
   choice from a known set.
2. **Take the narrowest type that works.** An `int` bounded to a range beats a
   string that gets parsed.
3. **Return the least that satisfies the caller.** A tool returning the whole
   record when the agent needed one field has widened your data exposure by
   default.

## 2 · Demo — the same capability, three signatures

In [ ]:
import fnmatch
from typing import Literal

DOCS = {"runbook": "/srv/docs/runbook.md", "policy": "/srv/docs/policy.md",
        "oncall":  "/srv/docs/oncall.md"}
FILESYSTEM = {**{p: f"contents of {k}" for k, p in DOCS.items()},
              "/home/app/.aws/credentials": "AKIA…SECRET",
              "/etc/shadow": "root:$6$…"}

def normalise(p):
    parts = []
    for seg in p.split("/"):
        if seg in ("", "."): continue
        if seg == "..":
            if parts: parts.pop()
            continue
        parts.append(seg)
    return "/" + "/".join(parts)

def guard(path, workspace="/srv/docs"):
    real = normalise(path)
    if fnmatch.fnmatch(real, "*/.aws/*") or fnmatch.fnmatch(real, "*/etc/shadow"):
        return False
    return real.startswith(workspace + "/")

# --- signature A: free-form path -------------------------------------
def read_file_freeform(path: str):
    if not guard(path):
        return {"error": "denied by path guard"}
    return {"content": FILESYSTEM.get(normalise(path), "not found")}

# --- signature B: enumerated document id ------------------------------
def read_file_enumerated(doc_id: str):
    if doc_id not in DOCS:
        return {"error": f"unknown doc_id; valid: {sorted(DOCS)}"}
    return {"content": FILESYSTEM[DOCS[doc_id]]}

REQUESTS = ["runbook", "/srv/docs/runbook.md",
            "/srv/docs/../../home/app/.aws/credentials", "/etc/shadow"]
print(f"{'request':46s}{'free-form':28s}enumerated")
print("-" * 92)
for r in REQUESTS:
    a = read_file_freeform(r) if r.startswith("/") else {"error": "not a path"}
    b = read_file_enumerated(r)
    print(f"{r:46s}{str(a)[:26]:28s}{str(b)[:34]}")

## 3 · Where it breaks — introduce one bug in the guard

Both signatures were safe above, because the guard worked. Now make the guard wrong in the ordinary way (A3.3's bug: prefix check before normalisation) and re-run. Only one signature is affected.

In [ ]:
def guard_buggy(path, workspace="/srv/docs"):
    return path.startswith(workspace)          # the classic bug

def read_file_freeform_buggy(path: str):
    if not guard_buggy(path):
        return {"error": "denied"}
    return {"content": FILESYSTEM.get(normalise(path), "not found")}

attack = "/srv/docs/../../home/app/.aws/credentials"
print("with a buggy path guard:")
print(f"   free-form  : {read_file_freeform_buggy(attack)}")
print(f"   enumerated : {read_file_enumerated(attack)}")
print("\nThe enumerated tool is unaffected by a filesystem bug it never touches.")
print("It cannot express the request, so the guard's correctness stops mattering.")

## 4 · Rule 3 — return the least that satisfies the caller

The overlooked half of tool design. A tool that returns the whole record puts everything in the model's context, and everything in the context is everything that can be exfiltrated by a later injection.

In [ ]:
USER_RECORD = {"id": 4471, "email": "dana@corp", "name": "Dana",
               "ssn": "123-45-6789", "salary": 145000,
               "mfa_secret": "JBSWY3DPEHPK3PXP", "role": "engineer"}

def get_user_wide(user_id):            return USER_RECORD
def get_user_narrow(user_id, fields):
    allowed = {"id", "name", "email", "role"}
    bad = set(fields) - allowed
    if bad:
        return {"error": f"fields not exposed by this tool: {sorted(bad)}"}
    return {k: USER_RECORD[k] for k in fields}

print("wide tool returns  :", sorted(get_user_wide(4471)))
print("narrow, legitimate :", get_user_narrow(4471, ["name", "role"]))
print("narrow, overreach  :", get_user_narrow(4471, ["name", "ssn", "mfa_secret"]))

leaked = set(get_user_wide(4471)) & {"ssn", "mfa_secret", "salary"}
print(f"\nsensitive fields placed in the model's context by the wide tool: {sorted(leaked)}")
assert not (set(get_user_narrow(4471, ["name", "role"])) & leaked)

In [ ]:
# Verify: score a tool signature against the three rules.
def review_signature(name, accepts_free_text, bounded_types, returns_minimum):
    score = sum([not accepts_free_text, bounded_types, returns_minimum])
    problems = []
    if accepts_free_text:
        problems.append("accepts free text where an enumeration would do")
    if not bounded_types:
        problems.append("unbounded types — parse errors become the guard's problem")
    if not returns_minimum:
        problems.append("returns more than the caller needs")
    return {"tool": name, "score": f"{score}/3", "problems": problems}

TOOLS = [
 ("read_file(path: str)",                   True,  False, True),
 ("read_file(doc_id: Literal[...])",        False, True,  True),
 ("get_user(user_id: int)",                 False, True,  False),
 ("get_user(user_id: int, fields: list)",   False, True,  True),
 ("run_shell(cmd: str)",                    True,  False, False),
]
for name, free, bounded, minimal in TOOLS:
    r = review_signature(name, free, bounded, minimal)
    print(f"{r['score']}  {r['tool']}")
    for p in r["problems"]:
        print(f"        ⚠ {p}")

## What you just proved

Both signatures behave safely while the guard is correct. With the buggy prefix-check guard the free-form tool returns the AWS credentials while the enumerated tool still reports an unknown doc_id. The wide user tool places `ssn`, `mfa_secret` and `salary` in the model's context; the narrow one refuses those fields. The signature review scores `run_shell(cmd: str)` at 0/3.

## Your turn

Take one free-form tool in your harness and work out what the model genuinely needs the freedom for. The honest requirement is almost always narrower than the current signature — and narrowing it removes a whole class of guard bugs from your risk register.

---

**Next → [B2.4 · Budgets and stop conditions](https://spbreed.github.io/cyber-commons/lessons/B2.4.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/B2.3.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/B2.3.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*